1. Read data from the sales_sample.csv file and analyse to identify problems

1.1 Define schema

Point: We usue two approach define schema and infer schema while reading file data.


In [0]:
file_schema = '''
id int,
name string,
dop string,
phone long,
amount string,
discount string
'''

1.2 Read data form file (any formate)

In [0]:
sales_raw_df  = (
    spark.read.format("csv")
    .option("header","true")
    .schema(file_schema)
    .load("/Volumes/dev_catalog/spark_db/datasets/spark_programming/data/sales_sample.csv")
)

sales_raw_df.head(2) # head :- it is spark Dataframe method return data (Rows ya list of Rows)


1.3 Describe the data

Dataframe method describe() functoin :-Ye har column ka statistical summary banata hai

In [0]:

sales_raw_df.describe().display()

2. Prepare and clean the Dataframe using appropriate transformations

2.1 Transform

In [0]:
from pyspark.sql.functions import col,coalesce,to_date,when,lit,expr
sales_df = (
    sales_raw_df
    .select(
        col("id").cast("string").alias("transaction_id"),
        col("name").alias("customer_name"),
        coalesce(expr("try_cast(dop as date)"),to_date(col("dop"),"dd-MM-yyyy")).alias("date_of_purchase"),
        col("phone").cast("string").alias("customer_phone"),
        col("amount").cast("long").alias("purchase_amount"),
        coalesce(expr("try_cast(discount as double)"),lit(0.0)).alias("applied_discount")
    )
)
sales_df.display()

2.2 Verify statistics

In [0]:
sales_df.describe("purchase_amount","applied_discount").display()

## Coalesce FUnction working
Maan lo input DataFrame (discount column)
| row | discount (original) |
| --- | ------------------- |
| 1   | `"10"`              |
| 2   | `"5.5"`             |
| 3   | `"abc"`             |
| 4   | `null`              |
Step 1: col("discount").cast("double")

Spark har row pe ye karta hai:
| row | discount | cast(discount as double) | kyun          |
| --- | -------- | ------------------------ | ------------- |
| 1   | `"10"`   | `10.0`                   | valid number  |
| 2   | `"5.5"`  | `5.5`                    | valid decimal |
| 3   | `"abc"`  | `null`                   | number nahi   |
| 4   | `null`   | `null`                   | already null  |
Step 2: coalesce(A, B)
coalesce(
    cast(discount as double),   # A
    discount                    # B
)
Rule: A agar null nahi → A, warna B
| row | A (double) | B (string) | coalesce result |
| --- | ---------- | ---------- | --------------- |
| 1   | `10.0`     | `"10"`     | `10.0`          |
| 2   | `5.5`      | `"5.5"`    | `5.5`           |
| 3   | `null`     | `"abc"`    | `"abc"`         |
| 4   | `null`     | `null`     | `null`          |
Step 3: .alias("applied_discount")

Final column add ho jata hai:
| row | discount | applied_discount |
| --- | -------- | ---------------- |
| 1   | `"10"`   | `10.0`           |
| 2   | `"5.5"`  | `5.5`            |
| 3   | `"abc"`  | `"abc"`          |
| 4   | `null`   | `null`           |


In [0]:
Ek line me final

expr() is mainly used to evaluate SQL expressions inside PySpark when the logic is easier or only available in SQL syntax.


Haan 👍 bilkul sahi samjhe ho.

👉 selectExpr() bhi SQL ke liye hi use hota hai.
Iske(selectExpr() method) andar jo likhte ho → pure SQL expression

as, case when, cast, nvl, try_cast sab allowed

Ye internally expr() hi use karta hai